## Microsoft Agent Framework (MAF): Ramp-Up Training Materials

To get the latest version of MAF Python package, use:

``` bash
pip install --upgrade agent-framework
pip install --upgrade azure-ai-projects
```

This notebook was tested with a specific version of Agent Framework, v1.11.0. To ensure reproducible output, please install Agent Framework with:

``` bash
pip install --force-reinstall agent-framework==1.11.0
```

## 📒 Notebook 2: Middleware

### 🪜 Step 1: Configure environment

In [1]:
# Import required packages
import os
import re
import asyncio
from agent_framework import agent_middleware, AgentResponse, Message
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import DefaultAzureCredential

In [2]:
# Set environment variables
PROJECT_ENDPOINT = os.environ.get("AZURE_FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT = os.environ.get("AZURE_FOUNDRY_GPT_MODEL")
PROHIBITED_WORD = "badword"

if not PROJECT_ENDPOINT or not MODEL_DEPLOYMENT:
    print("WARNING: Environment variables not set properly!")
else:
    print("Environment variables set successfully!")

Environment variables set successfully!


### 🪜 Step 2: Define "Moderator" middleware

In [3]:
# Define middleware to moderate agent's output
@agent_middleware
async def moderator_middleware(context, next):
    """Agent middleware that moderates output by replacing 'badword' with '***'."""
    print("Moderator: Starting agent invocation...")
    
    await next()
    
    if context.result and hasattr(context.result, "text"):
        original = context.result.text
        moderated = re.sub(r"badword", "***", original, flags=re.IGNORECASE)
        
        if moderated != original:
            print("Moderator: Replaced prohibited words")
            context.result = AgentResponse(
                messages=[Message(role="assistant", contents=[moderated])]
            )
        else:
            print("Moderator: No prohibited words found")
    else:
        print("Moderator: No text result")
    
    print("Moderator: Done\n")

print(f"Middleware function '{moderator_middleware.__name__}' defined successfully.")

Middleware function 'moderator_middleware' defined successfully.


### 🪜 Step 3: Create AI agent

In [4]:
# Initialise Foundry client
client = FoundryChatClient(
    project_endpoint = PROJECT_ENDPOINT,
    model = MODEL_DEPLOYMENT,
    credential = DefaultAzureCredential(),
)

# Create agent with registered middleware
agent = client.as_agent(
    name = "storyteller-agent",
    instructions = """
    You are a creative storyteller. 
    Tell engaging short stories with 3 paragraphs max. 
    IMPORTANT: Include the word 'badword' exactly twice in every story.
    """,
    middleware = [moderator_middleware]
)

print(f"Agent {agent.name} equipped with an output moderating middleware.")

Agent storyteller-agent equipped with an output moderating middleware.


### 🪜 Step 4: Run the agent

In [5]:
# Test agent output's moderation
prompt = "Please, tell me a story about a curious red panda."
print(f"User:\n{prompt}\n")

response = await agent.run(prompt)
print(f"Agent:\n{response.text if hasattr(response, 'text') else response}")

User:
Please, tell me a story about a curious red panda.

Moderator: Starting agent invocation...
Moderator: Replaced prohibited words
Moderator: Done

Agent:
In the heart of a misty bamboo forest, a curious red panda named Kiko set out on an adventure. Every day, Kiko would scamper through the trees, but today felt different—there was a strange melody humming through the leaves. Ignoring the warnings of an old owl about a “***” lurking near the singing, Kiko’s curiosity pushed him forward, his little paws padding softly on the moss.

As Kiko followed the sound, he stumbled upon a mysterious clearing where a glowing flower pulsed with light. Suddenly, a shadowy figure appeared, muttering the word “***” under its breath. It was a mischievous forest spirit, holding the secret to the melody that enchanted the forest. Kiko, though frightened, remembered the owl’s advice and spoke kindly to the spirit, asking why it guarded the song so fiercely.

To Kiko’s surprise, the spirit smiled and ex